In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd
import numpy as np
import unicodedata

# Advanced Stats Exploration

Goal: check whether richer player statistics can improve valuation modelling,
especially for defenders, midfielders, and goalkeepers.

This notebook does not change the main model yet. It only answers:

1. What advanced player stats are available?
2. Do they cover Premier League seasons we care about?
3. Can they be merged reliably with our existing player-season data?
4. Which columns are useful enough to become V2 features?

In [2]:
advanced_stats_path = project_root / "data" / "raw" / "players_data-2025_2026.csv"

advanced_stats = pd.read_csv(advanced_stats_path)

advanced_stats.shape


(2839, 102)

In [3]:
advanced_stats.columns.tolist()

['Rk',
 'Player',
 'Nation',
 'Pos',
 'Squad',
 'Comp',
 'Age',
 'Born',
 'MP',
 'Starts',
 'Min',
 '90s',
 'Gls',
 'Ast',
 'G+A',
 'G-PK',
 'PK',
 'PKatt',
 'CrdY',
 'CrdR',
 'G+A-PK',
 'Rk_stats_keeper',
 'Nation_stats_keeper',
 'Pos_stats_keeper',
 'Comp_stats_keeper',
 'Age_stats_keeper',
 'Born_stats_keeper',
 'MP_stats_keeper',
 'Starts_stats_keeper',
 'Min_stats_keeper',
 '90s_stats_keeper',
 'GA',
 'GA90',
 'SoTA',
 'Saves',
 'Save%',
 'W',
 'D',
 'L',
 'CS',
 'CS%',
 'PKatt_stats_keeper',
 'PKA',
 'PKsv',
 'PKm',
 'Rk_stats_shooting',
 'Nation_stats_shooting',
 'Pos_stats_shooting',
 'Comp_stats_shooting',
 'Age_stats_shooting',
 'Born_stats_shooting',
 '90s_stats_shooting',
 'Gls_stats_shooting',
 'Sh',
 'SoT',
 'SoT%',
 'Sh/90',
 'SoT/90',
 'G/Sh',
 'G/SoT',
 'PK_stats_shooting',
 'PKatt_stats_shooting',
 'Rk_stats_playing_time',
 'Nation_stats_playing_time',
 'Pos_stats_playing_time',
 'Comp_stats_playing_time',
 'Age_stats_playing_time',
 'Born_stats_playing_time',
 'MP_st

In [4]:
pl_advanced_stats = advanced_stats.loc[
    advanced_stats["Comp"].eq("eng Premier League")
].copy()

pl_advanced_stats.shape, pl_advanced_stats["Player"].nunique()


((551, 102), 537)

In [5]:
sorted(pl_advanced_stats["Squad"].unique())

['Arsenal',
 'Aston Villa',
 'Bournemouth',
 'Brentford',
 'Brighton',
 'Burnley',
 'Chelsea',
 'Crystal Palace',
 'Everton',
 'Fulham',
 'Leeds United',
 'Liverpool',
 'Manchester City',
 'Manchester Utd',
 'Newcastle United',
 'Nottingham Forest',
 'Sunderland',
 'Tottenham Hotspur',
 'West Ham United',
 'Wolves']

In [6]:
scoring_results = pd.read_csv(
    project_root / "data" / "processed" / "scoring_results_2025_26.csv"
)

sorted(scoring_results["season_club_name"].dropna().unique())

['AFC Bournemouth',
 'Arsenal FC',
 'Aston Villa',
 'Brentford FC',
 'Brighton & Hove Albion',
 'Burnley FC',
 'Chelsea FC',
 'Crystal Palace',
 'Everton FC',
 'Fulham FC',
 'Leeds United',
 'Liverpool FC',
 'Manchester City',
 'Manchester United',
 'Newcastle United',
 'Nottingham Forest',
 'Sunderland AFC',
 'Tottenham Hotspur',
 'West Ham United',
 'Wolverhampton Wanderers']

In [7]:
fbref_club_to_transfermarkt_club = {
    "Arsenal": "Arsenal FC",
    "Aston Villa": "Aston Villa",
    "Bournemouth": "AFC Bournemouth",
    "Brentford": "Brentford FC",
    "Brighton": "Brighton & Hove Albion",
    "Burnley": "Burnley FC",
    "Chelsea": "Chelsea FC",
    "Crystal Palace": "Crystal Palace",
    "Everton": "Everton FC",
    "Fulham": "Fulham FC",
    "Leeds United": "Leeds United",
    "Liverpool": "Liverpool FC",
    "Manchester City": "Manchester City",
    "Manchester Utd": "Manchester United",
    "Newcastle United": "Newcastle United",
    "Nottingham Forest": "Nottingham Forest",
    "Sunderland": "Sunderland AFC",
    "Tottenham Hotspur": "Tottenham Hotspur",
    "West Ham United": "West Ham United",
    "Wolves": "Wolverhampton Wanderers",
}

pl_advanced_stats["season_club_name"] = pl_advanced_stats["Squad"].map(
    fbref_club_to_transfermarkt_club
)

pl_advanced_stats["season_club_name"].isna().sum()

np.int64(0)

In [8]:
advanced_merge_test = scoring_results.merge(
    pl_advanced_stats,
    left_on=["player_name", "season_club_name"],
    right_on=["Player", "season_club_name"],
    how="left",
    indicator=True,
)

advanced_merge_test["_merge"].value_counts()

_merge
both          493
left_only      44
right_only      0
Name: count, dtype: int64

In [9]:
advanced_merge_test.loc[
    advanced_merge_test["_merge"].eq("left_only"),
    [
        "player_name",
        "season_club_name",
        "position",
        "minutes_played",
        "current_market_value_in_eur",
    ],
].sort_values("minutes_played", ascending=False).head(30)

,player_name,season_club_name,position,minutes_played,current_market_value_in_eur
255,Djordje Petrovic,AFC Bournemouth,Goalkeeper,3420,28000000.0
175,Ferdi Kadıoğlu,Brighton & Hove Albion,Defender,3134,28000000.0
241,Gabriel,Arsenal FC,Defender,2751,75000000.0
415,Yegor Yarmolyuk,Brentford FC,Midfield,2667,25000000.0
428,Nico O'Reilly,Manchester City,Defender,2649,40000000.0
152,Ladislav Krejci,Wolverhampton Wanderers,Defender,2361,22000000.0
208,Ismaïla Sarr,Crystal Palace,Attack,2177,35000000.0
294,Dan Ballard,Sunderland AFC,Defender,2148,NaN
167,Florentino,Burnley FC,Midfield,2114,22000000.0
29,Idrissa Gueye,Everton FC,Midfield,2102,1000000.0


## Name matching strategy

The first exact merge misses players because Transfermarkt and the advanced-stats file use different spellings, accents, shortened names, and nickname forms.

We handle this in two stages:

1. normalise names mechanically, removing accents/punctuation where safe;
2. apply a small explicit alias map for genuine name differences.

This keeps the matching logic reviewable instead of hiding it inside fuzzy matching.


In [10]:
SPECIAL_CHARACTER_REPLACEMENTS = {
    "đ": "d",
    "Đ": "D",
    "ı": "i",
    "ł": "l",
    "Ł": "L",
    "ø": "o",
    "Ø": "O",
    "þ": "th",
    "Þ": "Th",
    "æ": "ae",
    "Æ": "Ae",
    "œ": "oe",
    "Œ": "Oe",
}


def normalise_player_name(name):
    if pd.isna(name):
        return name

    for original, replacement in SPECIAL_CHARACTER_REPLACEMENTS.items():
        name = name.replace(original, replacement)

    normalised = unicodedata.normalize("NFKD", name)
    normalised = "".join(
        character for character in normalised
        if not unicodedata.combining(character)
    )

    normalised = normalised.lower()
    normalised = normalised.replace("'", "")
    normalised = normalised.replace("’", "")
    normalised = normalised.replace("-", " ")
    normalised = " ".join(normalised.split())

    return normalised


In [11]:
manual_player_name_key_corrections = {
    ("djordje petrovic", "AFC Bournemouth"): "dorde petrovic",
    ("junior kroupi", "AFC Bournemouth"): "eli junior kroupi",
    ("hamed traore", "AFC Bournemouth"): "hamed junior traore",
    ("gabriel", "Arsenal FC"): "gabriel magalhaes",
    ("emiliano buendia", "Aston Villa"): "emi buendia",
    ("alysson", "Aston Villa"): "alysson edward",
    ("jamaldeen jimoh aloba", "Aston Villa"): "jamaldeen jimoh",
    ("yegor yarmolyuk", "Brentford FC"): "yehor yarmoliuk",
    ("florentino", "Burnley FC"): "florentino luis",
    ("hannibal", "Burnley FC"): "hannibal mejbri",
    ("jaydon banel", "Burnley FC"): "jaydon amauri banel",
    ("lucas pires", "Burnley FC"): "lucas",
    ("estevao", "Chelsea FC"): "estevao willian",
    ("josh acheampong", "Chelsea FC"): "joshua acheampong",
    ("yeremy pino", "Crystal Palace"): "yeremi pino",
    ("idrissa gueye", "Everton FC"): "idrissa gana gueye",
    ("josh king", "Fulham FC"): "joshua king",
    ("nico gonzalez", "Manchester City"): "nicolas gonzalez",
    ("savinho", "Manchester City"): "savio",
    ("john victor", "Nottingham Forest"): "john",
    ("dan ballard", "Sunderland AFC"): "daniel ballard",
    ("tino livramento", "Newcastle United"): "valentino livramento",
    ("maximilian kilman", "West Ham United"): "max kilman",
    ("taty castellanos", "West Ham United"): "valentin castellanos",
    ("andrew irving", "West Ham United"): "andy irving",
    ("igor julio", "West Ham United"): "igor",
    ("hee chan hwang", "Wolverhampton Wanderers"): "hwang hee chan",
    ("toti", "Wolverhampton Wanderers"): "toti gomes",
}

scoring_results["player_name_key"] = scoring_results["player_name"].apply(
    normalise_player_name
)
pl_advanced_stats["player_name_key"] = pl_advanced_stats["Player"].apply(
    normalise_player_name
)

scoring_results["player_name_key_corrected"] = scoring_results.apply(
    lambda row: manual_player_name_key_corrections.get(
        (row["player_name_key"], row["season_club_name"]),
        row["player_name_key"],
    ),
    axis=1,
)
pl_advanced_stats["player_name_key_corrected"] = pl_advanced_stats[
    "player_name_key"
]

pl_advanced_stats.duplicated(
    ["player_name_key_corrected", "season_club_name"]
).sum()


np.int64(0)

## Corrected merge quality


In [12]:
advanced_stats_merged = scoring_results.merge(
    pl_advanced_stats,
    on=["player_name_key_corrected", "season_club_name"],
    how="left",
    indicator=True,
    suffixes=("", "_advanced"),
    validate="one_to_one",
)

advanced_stats_merged["_merge"].value_counts()


_merge
both          537
left_only       0
right_only      0
Name: count, dtype: int64

In [13]:
remaining_unmatched_players = advanced_stats_merged.loc[
    advanced_stats_merged["_merge"].eq("left_only"),
    [
        "player_name",
        "season_club_name",
        "position",
        "minutes_played",
        "current_market_value_in_eur",
    ],
].sort_values("minutes_played", ascending=False)

remaining_unmatched_players


,player_name,season_club_name,position,minutes_played,current_market_value_in_eur


In [14]:
merge_quality_summary = pd.DataFrame({
    "metric": [
        "scoring_rows",
        "advanced_pl_rows",
        "matched_rows",
        "unmatched_scoring_rows",
        "match_rate",
    ],
    "value": [
        len(scoring_results),
        len(pl_advanced_stats),
        int(advanced_stats_merged["_merge"].eq("both").sum()),
        int(advanced_stats_merged["_merge"].eq("left_only").sum()),
        advanced_stats_merged["_merge"].eq("both").mean(),
    ],
})

merge_quality_summary


,metric,value
0,scoring_rows,537.0
1,advanced_pl_rows,551.0
2,matched_rows,537.0
3,unmatched_scoring_rows,0.0
4,match_rate,1.0


## Available advanced columns

The full Kaggle file is richer than the light file. It adds goalkeeper, shooting, playing-time, and miscellaneous FBref-style sections. It still does not include every feature we wanted — notably progressive passing/carrying, xG/xAG, clearances, and blocks are absent — but it does add useful on-pitch team impact columns such as `Min%`, `PPM`, `onG`, `onGA`, `+/-`, and `On-Off`.


In [15]:
candidate_advanced_feature_columns = [
    "MP",
    "Starts",
    "Min",
    "90s",
    "Min%",
    "Mn/MP",
    "Mn/Start",
    "Compl",
    "Subs",
    "PPM",
    "onG",
    "onGA",
    "+/-",
    "+/-90",
    "On-Off",
    "Gls",
    "Ast",
    "Sh",
    "SoT",
    "SoT%",
    "Sh/90",
    "SoT/90",
    "Crs",
    "TklW",
    "Int",
    "Fld",
    "Fls",
    "Off",
    "OG",
    "GA",
    "GA90",
    "SoTA",
    "Saves",
    "Save%",
    "CS",
    "CS%",
]
available_candidate_columns = [
    column for column in candidate_advanced_feature_columns
    if column in advanced_stats_merged.columns
]

advanced_stats_merged[
    [
        "player_name",
        "season_club_name",
        "position",
        "minutes_played",
        *available_candidate_columns,
    ]
].head(10)


,player_name,season_club_name,position,minutes_played,MP,Starts,Min,90s,Min%,Mn/MP,...,Fls,Off,OG,GA,GA90,SoTA,Saves,Save%,CS,CS%
0,James Milner,Brighton & Hove Albion,Midfield,778,20,7,784,8.7,22.9,39,...,15,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Casemiro,Manchester United,Midfield,2589,34,33,2571,28.6,75.2,76,...,46,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Mateo Kovacic,Manchester City,Midfield,126,6,1,130,1.4,3.8,22,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Jordan Henderson,Brentford FC,Midfield,1921,32,22,1919,21.3,56.1,60,...,13,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Adam Smith,AFC Bournemouth,Defender,1080,22,14,1077,12.0,31.5,49,...,19,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Ashley Barnes,Burnley FC,Attack,41,7,0,47,0.5,1.4,7,...,2,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Danny Welbeck,Brighton & Hove Albion,Attack,2266,37,26,2258,25.1,66.0,61,...,31,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Séamus Coleman,Everton FC,Defender,20,5,1,21,0.2,0.6,4,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Bernd Leno,Fulham FC,Goalkeeper,3420,38,38,3420,38.0,100.0,90,...,0,0,1,51.0,1.34,146.0,98.0,67.1,9.0,23.7
9,Martin Dúbravka,Burnley FC,Goalkeeper,3150,35,35,3150,35.0,92.1,90,...,0,0,0,71.0,2.03,192.0,127.0,66.1,4.0,11.4


In [16]:
advanced_stats_merged[available_candidate_columns].isna().mean().sort_values()


MP          0.000000
OG          0.000000
Off         0.000000
Fls         0.000000
Fld         0.000000
Int         0.000000
TklW        0.000000
Crs         0.000000
SoT/90      0.000000
Sh/90       0.000000
SoT         0.000000
Ast         0.000000
Gls         0.000000
Sh          0.000000
Mn/MP       0.000000
+/-         0.000000
onGA        0.000000
Starts      0.000000
onG         0.000000
Min         0.000000
PPM         0.000000
Subs        0.000000
Compl       0.000000
90s         0.000000
+/-90       0.000000
Min%        0.000000
On-Off      0.009311
Mn/Start    0.117318
SoT%        0.175047
CS          0.925512
GA          0.925512
GA90        0.925512
SoTA        0.925512
Saves       0.925512
Save%       0.927374
CS%         0.927374
dtype: float64

## Initial conclusion

The 2025/26 light advanced-stats file can be merged cleanly into the current scoring population after normalisation plus explicit aliases.

For modelling, this file alone is not enough because it only covers 2025/26. To train properly, we need historical advanced-stat seasons using the same or compatible schema. The next data task is to find/download historical FBref-style files, then reuse this matching approach.


## Historical advanced-stat feature table

The separate historical Kaggle dataset is used for V2 training exploration. It covers earlier seasons, so unlike the 2025/26-only file it can be used to train and validate advanced-stat features.

The build script keeps FBref-style performance features and removes FIFA/EA value, wage, overall, potential, and attribute-rating columns.


In [17]:
from prem_valuation.advanced_stats import load_advanced_player_seasons

advanced_player_seasons = load_advanced_player_seasons()

advanced_player_seasons.shape


(24778, 99)

In [18]:
advanced_player_seasons.groupby("season_start_year").size()


season_start_year
2017    2949
2018    2872
2019    2983
2020    3218
2021    3374
2022    3370
2023    3629
2024    2383
dtype: int64

In [19]:
advanced_player_seasons["advanced_league_name"].value_counts()


advanced_league_name
ITA-Serie A               2983
ESP-La Liga               2964
TUR-Super Lig             2869
NED-Eredivisie            2842
ENG-Premier League        2806
POR-Primeira Liga         2667
GER-Bundesliga            2632
BEL-Jupiler Pro League    2510
FRA-Ligue 1               2505
Name: count, dtype: int64

In [20]:
advanced_player_seasons.columns.tolist()


['season_start_year',
 'season',
 'advanced_player_name',
 'advanced_team_name',
 'advanced_league_name',
 'advanced_nation',
 'advanced_position',
 'advanced_age',
 'advanced_birth_year',
 'advanced_mp',
 'advanced_starts',
 'advanced_min',
 'advanced_90s',
 'advanced_per90_crdy',
 'advanced_per90_crdr',
 'advanced_per90_prgc',
 'advanced_per90_prgp',
 'advanced_per90_prgr',
 'advanced_per90_gls',
 'advanced_per90_ast',
 'advanced_per90_g_plus_a',
 'advanced_per90_g_minus_pk',
 'advanced_per90_g_plus_a_minus_pk',
 'advanced_per90_xg',
 'advanced_per90_xag',
 'advanced_per90_xg_plus_xag',
 'advanced_per90_npxg',
 'advanced_per90_npxg_plus_xag',
 'advanced_sotpct',
 'advanced_sh_per_90',
 'advanced_sot_per_90',
 'advanced_g_per_sh',
 'advanced_g_per_sot',
 'advanced_dist',
 'advanced_expected_npxg_per_sh',
 'advanced_per90_total_cmp',
 'advanced_per90_total_att',
 'advanced_total_cmppct',
 'advanced_per90_total_totdist',
 'advanced_per90_total_prgdist',
 'advanced_per90_short_cmp',
 'ad